Scientific source notebook for the isolated reproduction tree. Calculation cells and saved execution objects are preserved. Narrative navigation has been adapted for this package. Use the reproduction runner and frozen-input prerequisites before executing. Historical evaluation blocks remain reused evidence; this copy does not establish independent replication.


# 08 · Forward stock-risk tests

**Question:** Does an extra characteristic improve subsequent risk forecasts?

Future residuals subtract formation-estimated FF5-plus-momentum predictions without refitting inside the future quarter. Cross-sectional association uses observed outcomes; forecasting trains only on earlier completed quarters, with train-only scaling and category construction.

Research report · Method and code guide · Evidence map


**Chapter route**

Measurement and interpretation principles · Data and design context · define the target, then forecast · Read and run this notebook · Calculation source: forward tests · Results, examples and interpretation · Spot-check a reported forecast improvement · Visual evidence · Findings · Analysis and takeaways · Bridge through the study


## Measurement and interpretation principles

Future residuals use formation-fixed FF5-plus-momentum coefficients; they are not refitted to the outcome quarter. The target is log annualized following-quarter volatility with at least 40 days. Expanding forecasts fit coefficients, scaling and categories only on earlier completed outcomes after eight training quarters; contemporaneous association is reported separately. Compare restricted and expanded models on identical companies, then average quarterly percentage MSE reductions.


## Data and design context

Fixed formation coefficients define future residual volatility. Association uses observed future outcomes; expanding forecasts use only earlier completed quarters. Paired comparisons share their eligible companies. Inference preserves quarter-level shocks and conditions on estimated exposures. See docs/findings/forward_tests.md for interpretation and the six-quarter confirmation limit.


## define the target, then forecast

`forward_outcomes` applies fixed formation coefficients. `evaluate_comparison` chooses a shared complete sample and trains only on previously completed outcomes. `fit_predict_linear` learns scaling and categories from training data; `summarize_comparisons` averages quarterly scores and quantifies time dependence. The display below reads saved estimates.


## Read and run this notebook

The calculation definitions now live here, alongside the purpose and interpretation. Use the chapter route for the argument and the bold function names for individual definitions. Definition cells do not download data or run the full study. The results/examples follow the definitions. Save edits and restart the kernel before a full rerun so shared imports read the saved version.


## Calculation source: forward tests

These are the actual definitions used by the study. Read the purpose above, then follow the functions below before the results. The small Python adapter imports only these marked cells; it does not run collection or display cells.


### Imports, conventions and measurement settings

Imports and fixed conventions used by the calculations below.


In [1]:
"""Fixed-coefficient future risk outcomes and matched chronological cross-sectional tests.

Formation-level associations are separate from expanding pooled forecasts. All paired model
comparisons share their observations, and forecast scaling and category maps use training data
only. Inference resamples whole quarterly scores, preserving each underlying cross-section.
"""

import sys

import numpy as np
import pandas as pd

from quiet_index_500 import settings
from quiet_index_500.foundation import calendar, loader
from quiet_index_500.research_io import save_table, make_notebook


standard_style_columns = ["m3_smb", "m3_hml", "m3_rmw", "m3_cma", "m3_mom"]
baseline_columns = ["m3_mkt_rf", "own_momentum", "trailing_residual_volatility"]
outcome_columns = ["log_forward_total_volatility", "log_forward_residual_volatility"]
factor_names = ["mkt_rf", "smb", "hml", "rmw", "cma", "mom"]


**forward_outcomes**

Future-quarter sample volatility, with formation model-3 coefficients held fixed.


In [2]:
def forward_outcomes(company_panel, daily_returns, factors):
    """Future-quarter sample volatility, with formation model-3 coefficients held fixed."""
    panel = company_panel.copy()
    panel["formation_date"] = pd.to_datetime(panel.formation_date)
    records = []
    for formation, companies in panel.groupby("formation_date", sort=True):
        if not calendar.following_quarter_complete(formation, factors.index.max()):
            continue
        sessions = calendar.following_quarter_sessions(daily_returns.index, formation)
        future_returns = daily_returns.reindex(sessions)
        future_factors = factors.reindex(sessions)
        design = np.column_stack([np.ones(len(sessions)), future_factors[factor_names].to_numpy()])
        for _, company in companies.iterrows():
            ticker_returns = future_returns[company.ticker]
            coefficients = company[["m3_alpha", *[f"m3_{factor}" for factor in factor_names]]].to_numpy(dtype=float)
            fixed_prediction = pd.Series(design @ coefficients, index=sessions)
            residual = ticker_returns - future_factors.rf - fixed_prediction
            total_days, residual_days = ticker_returns.notna().sum(), residual.notna().sum()
            total_vol = ticker_returns.std(ddof=1) * np.sqrt(settings.annualisation_sessions) if total_days >= settings.forward_outcome_minimum_valid_days else np.nan
            residual_vol = residual.std(ddof=1) * np.sqrt(settings.annualisation_sessions) if residual_days >= settings.forward_outcome_minimum_valid_days else np.nan
            records.append({"formation_date": formation, "ticker": company.ticker, "cik": company.cik,
                            "outcome_quarter": str(calendar.outcome_quarter(formation)),
                            "outcome_last_session": sessions.max(),
                            "outcome_end": calendar.outcome_quarter(formation).end_time.normalize(),
                            "outcome_block": calendar.outcome_block(formation),
                            "outcome_sessions": len(sessions), "forward_total_days": int(total_days),
                            "forward_residual_days": int(residual_days),
                            "forward_total_volatility": total_vol, "forward_residual_volatility": residual_vol,
                            "log_forward_total_volatility": np.log(total_vol) if total_vol > 0 else np.nan,
                            "log_forward_residual_volatility": np.log(residual_vol) if residual_vol > 0 else np.nan})
    return pd.DataFrame(records)


**comparison_specifications**

Predefined paired models; exploration is labelled and never chosen from outcomes.


In [3]:
def comparison_specifications(window_sessions):
    """Predefined paired models; exploration is labelled and never chosen from outcomes."""
    base = ["log_market_cap", *baseline_columns]
    styles = [*base, *standard_style_columns]
    semiconductor = [*styles, "m4_smh_excess"]
    specifications = [
        ("standard_styles", base, styles, "sector", "all", False),
        ("semiconductor", styles, semiconductor, "sector", "all", False),
        ("semiconductor_industry", styles, semiconductor, "industry", "all", False),
        ("liquidity", semiconductor, [*semiconductor, "log_amihud_252_per_million"], "sector", "all", False),
        ("size_interaction", semiconductor, [*semiconductor, "size_smh_interaction"], "sector", "all", False),
        ("semiconductor_no_size", [*baseline_columns, *standard_style_columns],
         [*baseline_columns, *standard_style_columns, "m4_smh_excess"], "sector", "all", False),
    ]
    if window_sessions == settings.main_window_sessions:
        specifications.extend([
            ("semiconductor_nonsemiconductors", styles, semiconductor, "sector", "nonsemiconductors", True),
            ("semiconductor_exclude_events", styles, semiconductor, "sector", "exclude_events", True),
            ("dollar_volume", semiconductor, [*semiconductor, "log_median_dollar_volume_60"], "sector", "all", True),
            ("semiconductor_variance_component", [*baseline_columns, *standard_style_columns],
             [*baseline_columns, *standard_style_columns, "semiconductor_variance_component"], "sector", "all", True),
        ])
    return [{"comparison": name, "restricted_columns": restricted, "extended_columns": extended,
             "controls": controls, "sample": sample, "exploratory": exploratory}
            for name, restricted, extended, controls, sample, exploratory in specifications]


**_category_labels**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [4]:
def _category_labels(data, control, allowed_industries=None):
    sector = data.gics_sector_code.astype(str)
    if control == "sector":
        return "sector_" + sector
    industry = data.gics_industry_group_code.astype(str)
    return ("industry_" + industry).where(industry.isin(allowed_industries), "sector_remainder_" + sector)


**fit_predict_linear**

OLS with scaling and category construction fitted only on the training frame.

The returns include coefficient rows on the standardized scale and the number of test
rows whose control category was absent from training. Such rows use the omitted-category
intercept rather than an invented category coefficient and are counted explicitly.


In [5]:
def fit_predict_linear(training, evaluation, numeric_columns, outcome, controls):
    """OLS with scaling and category construction fitted only on the training frame.

    The returns include coefficient rows on the standardized scale and the number of test
    rows whose control category was absent from training. Such rows use the omitted-category
    intercept rather than an invented category coefficient and are counted explicitly.
    """
    mean = training[numeric_columns].mean()
    scale = training[numeric_columns].std(ddof=0).replace(0, 1)
    allowed = None
    if controls == "industry":
        counts = training.groupby("gics_industry_group_code").cik.nunique()
        allowed = set(counts[counts >= settings.industry_group_minimum_companies].index.astype(str))
    train_labels = _category_labels(training, controls, allowed)
    test_labels = _category_labels(evaluation, controls, allowed)
    categories = sorted(train_labels.unique())
    def design(frame, labels):
        continuous = ((frame[numeric_columns] - mean) / scale).to_numpy(dtype=float)
        dummies = np.column_stack([labels.eq(category).to_numpy(dtype=float) for category in categories[1:]]) if len(categories) > 1 else np.empty((len(frame), 0))
        return np.column_stack([np.ones(len(frame)), continuous, dummies])
    train_design, test_design = design(training, train_labels), design(evaluation, test_labels)
    response = training[outcome].to_numpy(dtype=float)
    # the normal equations are small; a symmetric pseudoinverse also retains any sparse redundant controls
    gram = train_design.T @ train_design
    inverse = np.linalg.pinv(gram, hermitian=True)
    coefficients = inverse @ (train_design.T @ response)
    fitted = train_design @ coefficients
    residual = response - fitted
    rank = np.linalg.matrix_rank(gram)
    variance = residual @ residual / (len(training) - rank) if len(training) > rank else np.nan
    standard_error = np.sqrt(np.maximum(np.diag(inverse) * variance, 0))
    terms = ["const", *numeric_columns, *[f"control:{category}" for category in categories[1:]]]
    coefficient_rows = pd.DataFrame({"term": terms, "coefficient": coefficients, "conditional_ols_se": standard_error,
                                    "training_mean": [0, *mean.tolist(), *([0] * (len(categories) - 1))],
                                    "training_scale": [1, *scale.tolist(), *([1] * (len(categories) - 1))]})
    return {"prediction": test_design @ coefficients, "coefficients": coefficient_rows,
            "rank": int(rank), "parameters": train_design.shape[1],
            "unseen_control_rows": int((~test_labels.isin(categories)).sum()),
            "pooled_industry_rows": int(train_labels.str.startswith("sector_remainder_").sum()),
            "pooled_industry_groups": int(training.loc[train_labels.str.startswith("sector_remainder_"), "gics_industry_group_code"].nunique())}


**_metrics**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [6]:
def _metrics(actual, predicted):
    errors = actual - predicted
    total_variation = np.sum((actual - actual.mean()) ** 2)
    return {"mse": np.mean(errors ** 2), "mae": np.mean(np.abs(errors)),
            "r_squared": 1 - np.sum(errors ** 2) / total_variation if total_variation > 0 else np.nan,
            "spearman": pd.Series(actual).corr(pd.Series(predicted), method="spearman")}


**evaluate_comparison**

Paired per-quarter association and expanding forecast fits on identical samples.


In [7]:
def evaluate_comparison(data, specification, outcome):
    """Paired per-quarter association and expanding forecast fits on identical samples."""
    source = data.copy()
    if specification["sample"] == "nonsemiconductors":
        source = source[source.gics_sub_industry_code.ne(settings.mechanical_overlap_subindustry_code)]
    elif specification["sample"] == "exclude_events":
        source = source[~source.ticker.isin(settings.event_convention_sensitivity_tickers)]
    columns = specification["extended_columns"]
    sample = source.dropna(subset=[outcome, *columns, "gics_sector_code", "gics_industry_group_code"])
    quarterly_rows, coefficients, predictions = [], [], []
    for formation, test in sample.groupby("formation_date", sort=True):
        possible = source[source.formation_date.eq(formation)]
        historical = sample[(sample.outcome_last_session <= formation) & (sample.formation_date < formation)]
        training_quarters = historical.formation_date.nunique()
        for analysis in ["association", "forecast"]:
            training = test if analysis == "association" else historical
            if len(test) <= len(columns) + 12 or (analysis == "forecast" and training_quarters < settings.forecast_minimum_training_quarters):
                continue
            restricted = fit_predict_linear(training, test, specification["restricted_columns"], outcome, specification["controls"])
            extended = fit_predict_linear(training, test, columns, outcome, specification["controls"])
            actual = test[outcome].to_numpy(dtype=float)
            low_metrics, high_metrics = _metrics(actual, restricted["prediction"]), _metrics(actual, extended["prediction"])
            common = {"formation_date": formation, "outcome_block": test.outcome_block.iloc[0],
                      "analysis": analysis, "comparison": specification["comparison"], "outcome": outcome,
                      "controls": specification["controls"], "sample": specification["sample"],
                      "exploratory": specification["exploratory"]}
            quarterly_rows.append({**common, "companies": len(test), "possible_companies": len(possible),
                                   "excluded_missing": len(possible) - len(test),
                                   "represented_snapshot_weight": test.issuer_snapshot_weight.sum(),
                                   "training_rows": len(training), "training_quarters": 1 if analysis == "association" else training_quarters,
                                   "latest_training_outcome_end": training.outcome_end.max(),
                                   "latest_training_outcome_last_session": training.outcome_last_session.max(),
                                   "restricted_rank": restricted["rank"], "extended_rank": extended["rank"],
                                   "extended_parameters": extended["parameters"],
                                   "unseen_control_rows": extended["unseen_control_rows"],
                                   "pooled_industry_rows": extended["pooled_industry_rows"],
                                   "pooled_industry_groups": extended["pooled_industry_groups"],
                                   **{f"restricted_{name}": value for name, value in low_metrics.items()},
                                   **{f"extended_{name}": value for name, value in high_metrics.items()},
                                   "mse_improvement": low_metrics["mse"] - high_metrics["mse"],
                                   "relative_mse_improvement": 1 - high_metrics["mse"] / low_metrics["mse"],
                                   "r_squared_improvement": high_metrics["r_squared"] - low_metrics["r_squared"]})
            for model, fit in [("restricted", restricted), ("extended", extended)]:
                coefficients.append(fit["coefficients"].assign(**common, model=model))
            if analysis == "forecast":
                prediction_table = test[["formation_date", "ticker", "cik", "outcome_block"]].copy()
                prediction_table["comparison"] = specification["comparison"]
                prediction_table["outcome"] = outcome
                prediction_table["actual"] = actual
                prediction_table["restricted_prediction"] = restricted["prediction"]
                prediction_table["extended_prediction"] = extended["prediction"]
                predictions.append(prediction_table)
    return {"quarters": pd.DataFrame(quarterly_rows),
            "coefficients": pd.concat(coefficients, ignore_index=True) if coefficients else pd.DataFrame(),
            "predictions": pd.concat(predictions, ignore_index=True) if predictions else pd.DataFrame()}


**_block_resample_indices**

This helper supports the calculation sequence below; its arguments and return value are shown in the code.


In [8]:
def _block_resample_indices(starts, block_length, count, circular=False):
    indices = (starts[:, :, None] + np.arange(block_length)).reshape(len(starts), -1)[:, :count]
    return indices % count if circular else indices


**time_dependence_inference**

Mean, Newey-West uncertainty and moving-block bootstrap over ordered quarter scores.


In [9]:
def time_dependence_inference(values, seed=settings.bootstrap_random_seed):
    """Mean, Newey-West uncertainty and moving-block bootstrap over ordered quarter scores."""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    count = len(values)
    if count == 0:
        return {"quarters": 0}
    average = values.mean()
    centered = values - average
    long_run_variance = centered @ centered / count
    lags = min(settings.newey_west_lags, count - 1)
    for lag in range(1, lags + 1):
        weight = 1 - lag / (lags + 1)
        long_run_variance += 2 * weight * (centered[lag:] @ centered[:-lag]) / count
    standard_error = np.sqrt(max(long_run_variance, 0) / count)
    block_length = min(settings.bootstrap_block_length, count)
    generator = np.random.default_rng(seed)
    block_starts = generator.integers(0, count - block_length + 1,
                                     size=(settings.bootstrap_replications, int(np.ceil(count / block_length))))
    indices = _block_resample_indices(block_starts, block_length, count)
    bootstrapped_means = values[indices].mean(axis=1)
    circular_generator = np.random.default_rng(seed)
    circular_starts = circular_generator.integers(0, count,
                         size=(settings.bootstrap_replications, int(np.ceil(count / block_length))))
    circular_means = values[_block_resample_indices(circular_starts, block_length, count, circular=True)].mean(axis=1)
    return {"quarters": count, "mean": average, "newey_west_se": standard_error,
            "newey_west_t": average / standard_error if standard_error > 0 else np.nan,
            "bootstrap_low": np.quantile(bootstrapped_means, 0.025),
            "bootstrap_high": np.quantile(bootstrapped_means, 0.975),
            "circular_bootstrap_low": np.quantile(circular_means, 0.025),
            "circular_bootstrap_high": np.quantile(circular_means, 0.975),
            "positive_quarter_share": np.mean(values > 0), "newey_west_lags": lags,
            "bootstrap_block_length": block_length, "bootstrap_replications": settings.bootstrap_replications,
            "inference_conditions": "fixed estimated exposures; common quarter shocks preserved"}


**summarize_comparisons**

Inference by development, validation, confirmation and pooled evaluation blocks.


In [10]:
def summarize_comparisons(quarterly):
    """Inference by development, validation, confirmation and pooled evaluation blocks."""
    output = []
    for keys, group in quarterly.groupby(["window_sessions", "analysis", "comparison", "outcome", "exploratory"], sort=True):
        common = dict(zip(["window_sessions", "analysis", "comparison", "outcome", "exploratory"], keys))
        blocks = [("all", group), *list(group.groupby("outcome_block", sort=True))]
        for block, rows in blocks:
            rows = rows.sort_values("formation_date")
            for metric in ["mse_improvement", "relative_mse_improvement", "r_squared_improvement"]:
                output.append({**common, "outcome_block": block, "metric": metric,
                               "mean_companies": rows.companies.mean(),
                               **time_dependence_inference(rows[metric])})
    return pd.DataFrame(output)


**summarize_added_coefficients**

Quarterly mean association slopes for added terms on the within-quarter SD scale.


In [11]:
def summarize_added_coefficients(coefficients):
    """Quarterly mean association slopes for added terms on the within-quarter SD scale."""
    selected = coefficients[coefficients.analysis.eq("association") & coefficients.model.eq("extended")]
    term_by_comparison = {
        "semiconductor": ["m4_smh_excess"], "semiconductor_industry": ["m4_smh_excess"],
        "semiconductor_no_size": ["m4_smh_excess"], "semiconductor_nonsemiconductors": ["m4_smh_excess"],
        "semiconductor_exclude_events": ["m4_smh_excess"], "liquidity": ["log_amihud_252_per_million"],
        "dollar_volume": ["log_median_dollar_volume_60"], "size_interaction": ["size_smh_interaction"],
        "semiconductor_variance_component": ["semiconductor_variance_component"],
        "standard_styles": standard_style_columns,
    }
    output = []
    for keys, group in selected.groupby(["window_sessions", "comparison", "outcome", "term", "exploratory"], sort=True):
        common = dict(zip(["window_sessions", "comparison", "outcome", "term", "exploratory"], keys))
        if common["term"] not in term_by_comparison[common["comparison"]]:
            continue
        for block, rows in [("all", group), *list(group.groupby("outcome_block", sort=True))]:
            output.append({**common, "outcome_block": block,
                           "scale": "log volatility change per within-quarter characteristic SD, conditional on other columns",
                           **time_dependence_inference(rows.sort_values("formation_date").coefficient)})
    return pd.DataFrame(output)


**build_forward_tests**

Run the frozen forward tests, saving all statistics behind the findings.


In [12]:
def build_forward_tests():
    """Run the frozen forward tests, saving all statistics behind the findings."""
    returns, factors = loader.load_daily_returns(), loader.load_factors()
    all_quarters, all_coefficients, paths = [], [], {}
    freeze_path = settings.repository_root_folder / "docs/TEST_FREEZE.md"
    for window in [settings.main_window_sessions, settings.sensitivity_window_sessions]:
        panel_path = settings.licensed_results_folder / "panel" / f"company_panel_{window}.csv"
        panel = pd.read_csv(panel_path, parse_dates=["formation_date"])
        outcomes = forward_outcomes(panel, returns, factors)
        paths[f"outcomes_{window}"] = save_table(outcomes, "forward_tests", f"outcomes_{window}.csv",
                                                     inputs=[panel_path, freeze_path], source_modules=[settings.repository_root_folder / 'src/quiet_index_500/forward_tests.py'],
                                                     note="Future residuals use fixed formation FF5+momentum coefficients, no future refit; complete quarters only.")
        data = panel.merge(outcomes, on=["formation_date", "ticker", "cik"], validate="one_to_one")
        data["size_smh_interaction"] = data.log_market_cap * data.m4_smh_excess
        data["semiconductor_variance_component"] = data.m4_smh_excess ** 2 * data.smh_residual_volatility ** 2
        quarter_tables, coefficient_tables = [], []
        for outcome in outcome_columns:
            outcome_predictions = []
            for specification in comparison_specifications(window):
                result = evaluate_comparison(data, specification, outcome)
                quarter_tables.append(result["quarters"])
                coefficient_tables.append(result["coefficients"])
                outcome_predictions.append(result["predictions"])
            predictions = pd.concat(outcome_predictions, ignore_index=True).assign(window_sessions=window)
            suffix = outcome.removeprefix("log_forward_")
            paths[f"predictions_{window}_{suffix}"] = save_table(predictions, "forward_tests", f"predictions_{window}_{suffix}.csv",
                    licensed=True, inputs=[panel_path, paths[f"outcomes_{window}"], freeze_path], source_modules=[settings.repository_root_folder / 'src/quiet_index_500/forward_tests.py'],
                    note="Chronological paired predictions; all numeric scaling and control category maps fitted on earlier completed quarters only.")
        quarterly = pd.concat(quarter_tables, ignore_index=True).assign(window_sessions=window)
        coefficients = pd.concat(coefficient_tables, ignore_index=True).assign(window_sessions=window)
        for name, table in [("quarters", quarterly), ("coefficients", coefficients)]:
            paths[f"{name}_{window}"] = save_table(table, "forward_tests", f"{name}_{window}.csv", licensed=True,
                    inputs=[panel_path, paths[f"outcomes_{window}"], freeze_path], source_modules=[settings.repository_root_folder / 'src/quiet_index_500/forward_tests.py'],
                    note="Paired samples match exactly; association is separate from forecasting; company-level representatives only.")
        all_quarters.append(quarterly)
        all_coefficients.append(coefficients)
    summary = summarize_comparisons(pd.concat(all_quarters, ignore_index=True))
    paths["inference"] = save_table(summary, "forward_tests", "paired_inference.csv", licensed=True,
                                   inputs=[paths["quarters_504"], paths["quarters_252"], freeze_path], source_modules=[settings.repository_root_folder / 'src/quiet_index_500/forward_tests.py'],
                                   note="Whole-quarter score inference conditional on fixed exposures; six confirmation quarters do not justify broad significance claims.")
    paths["coefficient_inference"] = save_table(summarize_added_coefficients(pd.concat(all_coefficients, ignore_index=True)),
                    "forward_tests", "coefficient_inference.csv", licensed=True,
                    inputs=[paths["coefficients_504"], paths["coefficients_252"], freeze_path], source_modules=[settings.repository_root_folder / 'src/quiet_index_500/forward_tests.py'],
                    note="Association slopes on standardized feature scale; quarter NW/block uncertainty conditional on estimated exposures. Association fit gains are mechanically nonnegative.")
    make_notebook(settings.repository_root_folder / "notebooks/08_forward_risk_tests.ipynb", "Forward risk tests",
                  "Fixed formation coefficients define future residual volatility. Association uses observed future outcomes; expanding forecasts use only earlier completed quarters. Paired comparisons share their eligible companies. Inference preserves quarter-level shocks and conditions on estimated exposures. See docs/findings/forward_tests.md for interpretation and the six-quarter confirmation limit.",
                  ["from quiet_index_500.forward_tests import notebook_summary\nnotebook_summary()"])
    return paths


**notebook_summary**

Display the principal confirmation comparison from saved inference tables.


In [13]:
def notebook_summary():
    """Display the principal confirmation comparison from saved inference tables."""
    table = pd.read_csv(settings.licensed_results_folder / "forward_tests/paired_inference.csv")
    return table[table.comparison.isin(["semiconductor", "semiconductor_industry", "semiconductor_no_size"])
                 & table.outcome_block.eq("confirmation") & table.metric.eq("relative_mse_improvement")]


## Results, examples and interpretation

The cells below use the definitions above and the frozen study inputs.


In [14]:
notebook_summary()

,window_sessions,analysis,comparison,outcome,exploratory,outcome_block,metric,mean_companies,quarters,mean,...,newey_west_t,bootstrap_low,bootstrap_high,circular_bootstrap_low,circular_bootstrap_high,positive_quarter_share,newey_west_lags,bootstrap_block_length,bootstrap_replications,inference_conditions
28,252,association,semiconductor,log_forward_residual_volatility,False,confirmation,relative_mse_improvement,493.333333,6,0.004764,...,3.405789,0.001661,0.004764,0.001730,0.007867,1.000000,4,4,2000,fixed estimated exposures; common quarter shoc...
40,252,association,semiconductor,log_forward_total_volatility,False,confirmation,relative_mse_improvement,493.333333,6,0.003937,...,2.088769,0.001266,0.004189,0.001266,0.006609,1.000000,4,4,2000,fixed estimated exposures; common quarter shoc...
52,252,association,semiconductor_industry,log_forward_residual_volatility,False,confirmation,relative_mse_improvement,493.333333,6,0.001952,...,4.366194,0.000907,0.002311,0.000720,0.003184,1.000000,4,4,2000,fixed estimated exposures; common quarter shoc...
64,252,association,semiconductor_industry,log_forward_total_volatility,False,confirmation,relative_mse_improvement,493.333333,6,0.001770,...,2.410464,0.000761,0.001824,0.000761,0.002743,1.000000,4,4,2000,fixed estimated exposures; common quarter shoc...
76,252,association,semiconductor_no_size,log_forward_residual_volatility,False,confirmation,relative_mse_improvement,494.166667,6,0.004240,...,3.256591,0.001579,0.004240,0.001773,0.006900,1.000000,4,4,2000,fixed estimated exposures; common quarter shoc...
88,252,association,semiconductor_no_size,log_forward_total_volatility,False,confirmation,relative_mse_improvement,494.166667,6,0.003392,...,2.159914,0.001130,0.003507,0.001162,0.005655,1.000000,4,4,2000,fixed estimated exposures; common quarter shoc...
172,252,forecast,semiconductor,log_forward_residual_volatility,False,confirmation,relative_mse_improvement,493.333333,6,-0.006623,...,-2.209319,-0.007896,-0.001242,-0.012137,-0.001108,0.166667,4,4,2000,fixed estimated exposures; common quarter shoc...
184,252,forecast,semiconductor,log_forward_total_volatility,False,confirmation,relative_mse_improvement,493.333333,6,0.000652,...,0.201502,0.000652,0.009464,-0.008160,0.007554,0.500000,4,4,2000,fixed estimated exposures; common quarter shoc...
196,252,forecast,semiconductor_industry,log_forward_residual_volatility,False,confirmation,relative_mse_improvement,493.333333,6,0.000528,...,1.117704,-0.000375,0.000528,-0.000290,0.001431,0.666667,4,4,2000,fixed estimated exposures; common quarter shoc...
208,252,forecast,semiconductor_industry,log_forward_total_volatility,False,confirmation,relative_mse_improvement,493.333333,6,0.000155,...,0.139308,0.000155,0.003232,-0.002923,0.002603,0.500000,4,4,2000,fixed estimated exposures; common quarter shoc...


## Spot-check a reported forecast improvement

This cell reads saved study predictions and recomputes a paired forecast-error comparison; it does not fit a new specification. Compare its displayed result with the saved inference tables.


In [15]:
import pandas as pd
from quiet_index_500 import settings
pred = pd.read_csv(settings.licensed_results_folder / 'forward_tests/predictions_504_residual_volatility.csv')
sample = pred[pred.comparison.eq('semiconductor') & pred.formation_date.eq('2024-12-31')]
restricted_mse = ((sample.actual - sample.restricted_prediction)**2).mean()
extended_mse = ((sample.actual - sample.extended_prediction)**2).mean()
pd.Series({'companies': len(sample), 'restricted_mse': restricted_mse, 'extended_mse': extended_mse,
           'quarter_mse_reduction_percent': 100 * (1 - extended_mse / restricted_mse)})

companies                        489.000000
restricted_mse                     0.089128
extended_mse                       0.088760
quarter_mse_reduction_percent      0.413579
dtype: float64

## Visual evidence

Source figure: 04_forward_risk

These figures use saved results; see the figure guide for sources, sample definitions and limitations.


## Findings

Signed SMH lowers residual-risk forecast MSE by 1.04% in validation but raises it by 0.36% in confirmation. Industry controls do not rescue the latter. The Amihud extension improves confirmation by 2.76%; dollar volume improves it by 8.78% but is exploratory.

## Analysis and takeaways

The main residual-risk increment reverses from +1.04% validation improvement to −0.36% confirmation improvement; industry controls also deteriorate (−0.16%). Amihud improves confirmation MSE by 2.76%; dollar volume improves it by 8.78% and remains exploratory. These are changes in log-volatility forecast MSE, not reductions in risk. The size interaction does not restore the primary result; downside beta is not tested here. Six confirmation quarters and conditional first-stage inference limit confidence, including for favorable liquidity estimates.

## Bridge through the study

These results answer the stock-level question. Combine them with the separate portfolio evidence from 09 to write the final conclusion. Percentages are changes in log-volatility forecast error, not risk reduction.
